<a href="https://colab.research.google.com/github/whiplaaash/TF-NLP/blob/main/colab_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to NLP fundamentals in Tensor Flow

NLP has the goal of deriving information out of natural language (could be sequences, text or speach).

Another common term for NLP problems is sequence to seqence (seq2seq).

# Check for GPU (to change from CPU to GPU use "Srodowisko wykonawcze" and then "Zmien typ srodowiska wykonawczego").


In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-2c78490c-1e86-6b42-43b6-2fd2f59da9ed)


# Get helper functions

In [2]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py

--2026-11-08 23:07:27--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-11-08 23:07:27 (21.3 MB/s) - ‘helper_functions.py’ saved [10246/10246]



# Import a series of helper functions for the notebook.

In [3]:
from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

# Get a dataset

We are going to be using a dataset taken from Kaggle. It is an introduction to NLP dataset (text samples of Tweets labelled as a disaster or not a disaster).

Se the original source:
https://www.kaggle.com/competitions/nlp-getting-started

In [4]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip


--2026-11-08 23:07:31--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.2.207, 74.125.137.207, 2607:f8b0:4023:c03::cf, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.2.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip’

nlp_getting_started 100%[===================>] 593.11K  --.-KB/s    in 0.007s  

2026-11-08 23:07:32 (79.4 MB/s) - ‘nlp_getting_started.zip’ saved [607343/607343]



In [5]:
# Unzip data
unzip_data("nlp_getting_started.zip")

# Visualizing a text dataset

In [6]:
# We use pandas package to read the data.

import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
train_df.head()


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [7]:
train_df["text"][0]

'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'

In [8]:
# Shuffle training dataframe

train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [9]:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [10]:
# How many examples of class are there?
train_df_shuffled.target.value_counts()

# Data are quite well balanced (60:40)

,count
target,
0,4342
1,3271


In [11]:
# How many total samples?
len(train_df), len(test_df)

(7613, 3263)

# Let's visualize some data

In [12]:
# Let's generate 10 random samples
import random
random_index = random.randint(0, len(train_df)-5)
for row in train_df_shuffled[["text", "target"]][random_index:random_index+5].itertuples():
# we generate random samples in the range [random_index:random_index+5]
# itertuples() - returns values in the row as a tuple
  _, text, target = row
  print(f"Target: {target}", "(real disaster)" if target==1 else "(not real disaster)")
  print(f"Text:\n{text}\n")
  print("-----\n")

Target: 1 (real disaster)
Text:
.@CityofCalgary activates emergency plan amid severe thunderstorm warning http://t.co/pc7S8NxJ6Q #yyc #abstorm http://t.co/9xoHmMlMDY

-----

Target: 1 (real disaster)
Text:
White people I know you worry tirelessly about black on black crime but what are you going to do about your mass murderer problem?

-----

Target: 1 (real disaster)
Text:
Body shops inundated with cars dented by hail... Good news insurance pays... Bad news :  you are stuck with deductible !
#wcvb

-----

Target: 0 (not real disaster)
Text:
Err:509

-----

Target: 0 (not real disaster)
Text:
How do people bake without turning their kitchen into a war zone of eggs and flour

-----



In [13]:
# Split data into training and validation sets
from sklearn.model_selection import train_test_split

train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1, # use 10% of data as validation set
                                                                            random_state=42 # for reproduction of the data
                                                                            )

In [14]:
# Check lenghts
len(train_sentences), len(val_sentences), len(train_labels), len(val_labels)

(6851, 762, 6851, 762)

In [15]:
# check the first 10 samples
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

# Converting text to numbers (tokenization and embbeding)

Before building a model based on words, the first thing to do is to convert words to numbers.

**Tokenization**: converting tokens (words, subwords, characters) into numbers, usually with one-hot encoding.

**Embedding**: takes into account the context of w word, probability of beng close to other words. We may use pre-learnt embedding. For each token you create a matrix of feature vectors.

# Tokenization (text vectorization)

In [16]:
train_sentences[:5]

array(['@mogacola @zamtriossu i screamed after hitting tweet',
       'Imagine getting flattened by Kurt Zouma',
       '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
       "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
       'Somehow find you and I collide http://t.co/Ee8RpOahPk'],
      dtype=object)

In [29]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# Use the default parameters
text_vectorizer = TextVectorization(max_tokens=None, # how many tokens in the whole set of vocabulary
                                    standardize="lower_and_strip_punctuation",
                                    split="whitespace",
                                    ngrams=None, # groups of n number of words
                                    output_mode="int", # how to represent words as numbers
                                    output_sequence_length=None, # how long do you want your sequences to be
                                    pad_to_max_tokens=False)

In [30]:
# Find the avergae number of tokens in the training tweets
round(sum([len(i.split()) for i in train_sentences])/len(train_sentences))

15

In [31]:
# Setup text vectorization variables
max_vocab_length = 10000 # max number of words to have in our vocabulary
max_length = 15 # max length our sentences will be

# Use fixed, and no default, parameters
text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)